# 🤖 Enhanced ReAct Agent — Skills, Progressive Disclosure & Context Compaction

This notebook demonstrates how modern coding agents (Cursor, Claude Code, GitHub Copilot) work under the hood.

### The Big Idea
Instead of stuffing everything into one giant prompt, the agent:
1. Reads a **map.md** to discover available "skills" (capabilities)
2. Loads only the relevant skill file **on demand** (progressive disclosure)
3. Inside each skill, it finds **specific tools** it can call
4. When the conversation gets long, it **compacts context** using compact.md

This is a simplified but realistic model of what real agents do.

```
User Request
     │
     ▼
[Read map.md] ← always loaded (tiny, ~100 tokens)
     │
     ▼
[Decide: which skill is needed?]
     │
     ▼
[read_skill_file(skill_id)] ← load on demand (~500 tokens)
     │
     ▼
[Use tools from that skill: run_python / read_file]
     │
     ▼
[Answer user] ── if too long ──► [compact_context()]
```


## 1. Setup — Install & Import

In [11]:
# Both Gemini (primary) and OpenAI (backup) use the same openai Python client
!uv pip install -q openai python-dotenv

In [ ]:
import os
import json
import subprocess
from openai import OpenAI

# ══════════════════════════════════════════════════════════════
# STEP 1: Choose your LLM provider — uncomment ONE option below
# ══════════════════════════════════════════════════════════════

# ── Option A: OpenAI (requires OPENAI_API_KEY in .env or env var) ──
# from dotenv import load_dotenv; load_dotenv()
# MODEL  = "gpt-4o-mini"
# client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ── Option B: Google Colab + OpenAI ────────────────────────────────
# from google.colab import userdata
# MODEL  = "gpt-4o-mini"
# client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# ── Option C: Ollama — local, free (install from ollama.com) ───────
# Requires: ollama pull llama3.2  (or any model you prefer)
MODEL  = "qwen3:4b"   # change to any pulled model, e.g. phi3, smollm2
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

print(f"✓ Client ready  |  Model: {MODEL}")

✓ Client ready  |  Model: qwen3:4b


In [39]:
# Initialize tracking
conversation_log = []

## 2. Architecture Deep Dive: Understanding Tool Functions

### What Are Tools?

In a real agent, tools are the **ACTION space** — the only ways an agent can interact with the world.

Think of it like a chess engine that can only:
- Move a piece
- Evaluate a position  
- Request time analysis

In our agent, we limit tools to:
- **read_skill_file()** → Discover what the agent can do
- **read_file()** → Read data or code
- **run_python()** → Compute or test
- **compact_context()** → Manage memory

**Why limit tools?** Because keeping the model's "action space" small forces it to:
- Reason clearly about what to do
- Compose solutions from basic building blocks (like Unix pipes)
- Avoid hallucinating capabilities it doesn't have

### The Tool Schema

Each tool has a **JSON schema** that OpenAI understands. The model sees this schema and decides which tool to call. The schema includes:
- **name**: Short identifier
- **description**: Why the model should use this tool
- **parameters**: What arguments it needs

Let's define these tools:


In [28]:
# ──────────────────────────────────────────────────────────────────────────────
# PART 1: TOOL IMPLEMENTATIONS
# ──────────────────────────────────────────────────────────────────────────────

def read_skill_file(skill_id: str) -> str:
    """Load a skill file from the skills folder."""
    try:
        filepath = f"skills/{skill_id}.md"
        with open(filepath, 'r') as f:
            content = f.read()
        return f"[Skill: {skill_id}]\n{content}"
    except FileNotFoundError:
        return f"Error: Skill file 'skills/{skill_id}.md' not found."
    except Exception as e:
        return f"Error reading skill: {str(e)}"

def read_file(filepath: str) -> str:
    """Read a file from disk."""
    try:
        with open(filepath, 'r') as f:
            content = f.read()
        return content[:5000]  # Limit to first 5000 chars
    except FileNotFoundError:
        return f"Error: File '{filepath}' not found."
    except Exception as e:
        return f"Error reading file: {str(e)}"

def run_python(code: str) -> str:
    """Execute Python code safely with timeout."""
    import subprocess

    try:
        # Write code to temp file
        with open("_temp_code.py", "w") as f:
            f.write(code)

        # Run with timeout
        result = subprocess.run(
            ["python", "_temp_code.py"],
            capture_output=True,
            text=True,
            timeout=10
        )

        output = result.stdout
        if result.stderr:
            output += f"\n[stderr]: {result.stderr}"

        return output if output else "[Code executed successfully]"
    except subprocess.TimeoutExpired:
        return "Error: Code execution timed out (10 seconds)"
    except Exception as e:
        return f"Error executing code: {str(e)}"

def compact_context() -> str:
    """Get instructions for compacting context."""
    try:
        with open("skills/compact.md", 'r') as f:
            return f.read()
    except FileNotFoundError:
        return "Compact skill not found. Use this format:\nGoal: ...\nKey Decisions: ...\nWork Completed: ...\nCurrent State: ...\nNext Step: ..."
    except Exception as e:
        return f"Error: {str(e)}"


In [29]:
# ──────────────────────────────────────────────────────────────────────────────
# PART 2: TOOL SCHEMA (For OpenAI API)
# ──────────────────────────────────────────────────────────────────────────────

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "read_skill_file",
            "description": "Read a skill file to discover available tools and how to use them. Skills are markdown files in the skills/ folder.",
            "parameters": {
                "type": "object",
                "properties": {
                    "skill_id": {
                        "type": "string",
                        "description": "The ID of the skill (e.g., 'code_helper', 'data_analyst')"
                    }
                },
                "required": ["skill_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the contents of a file (e.g., code, data, config).",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {
                        "type": "string",
                        "description": "The path to the file to read"
                    }
                },
                "required": ["filepath"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_python",
            "description": "Execute Python code to compute results, test functions, or analyze data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "The Python code to execute"
                    }
                },
                "required": ["code"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "compact_context",
            "description": "Get instructions for compacting conversation history when context gets long. No arguments needed.",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    }
]

In [30]:
# ──────────────────────────────────────────────────────────────────────────────
# PART 3: TOOL DISPATCHER
# ──────────────────────────────────────────────────────────────────────────────

def dispatch_tool(tool_name: str, tool_args: dict) -> str:
    """Execute a tool and return its result."""
    if tool_name == "read_skill_file":
        return read_skill_file(tool_args["skill_id"])
    elif tool_name == "read_file":
        return read_file(tool_args["filepath"])
    elif tool_name == "run_python":
        return run_python(tool_args["code"])
    elif tool_name == "compact_context":
        return compact_context()
    else:
        return f"Unknown tool: {tool_name}"

In [31]:
# ──────────────────────────────────────────────────────────────────────────────
# PART 4: BUILD SYSTEM PROMPT
# ──────────────────────────────────────────────────────────────────────────────

def build_system_prompt() -> str:
    """Build the system prompt with ReAct rules and skill map."""

    # Load map.md (always available)
    try:
        with open("skills/map.md", 'r') as f:
            skill_map = f.read()
    except:
        skill_map = """Available Skills:
| Skill ID | Purpose | When to Use |
|---|---|---|
| code_helper | Write, test, and debug code | User asks for code, debugging, or testing |
| data_analyst | Analyze data and create visualizations | User asks about data analysis, statistics |
"""

    system_prompt = f"""You are an enhanced ReAct agent that demonstrates how modern coding agents (Claude Code, Cursor, GitHub Copilot) work.

## RULE 1 (MANDATORY): ALWAYS use progressive disclosure
**Before attempting ANY task**, you MUST call `read_skill_file()` with the appropriate skill ID.
This is not optional — it's the core of how agents discover capabilities.

## Available Skills (from skills/map.md):
{skill_map}

## Your Task: Follow the ReAct Pattern
For EVERY problem, follow this cycle exactly:

1. [Reflect] Analyze the user's request
2. [Plan] Decide which skill is needed  
3. [Tool Call] Call read_skill_file(skill_id) to load the skill
4. [Observe] Read what tools are available in the skill
5. [Act] Use the skill's tools (run_python, read_file, etc.)
6. [Respond] Provide the final answer

## Behavior Rules
- Show your reasoning: include [Reflect], [Plan], [Observe] in your thinking
- Be transparent: explain which skill you loaded and WHY
- Be thorough: use tool results to verify your work
- Think step-by-step: break complex problems into tool calls

## Constraints
- You can ONLY use these tools: read_skill_file, read_file, run_python, compact_context
- You can call tools multiple times per message
- Max 10 steps per task (prevents infinite loops)
- Always prioritize correct answers over speed
"""

    return system_prompt.strip()

# Build the system prompt once
system_prompt = build_system_prompt()

## 3. Build the System Prompt — The Agent's "Personality"

### What's a System Prompt?

The system prompt is the **constitutional document** for the agent:
- It defines behavior rules
- It includes always-available information (map.md)
- It sets expectations for reasoning style (ReAct format)
- It constrains what the agent can/cannot do

### Key Design Principle: Include Only What's Always Needed

❌ **Bad**: System prompt includes all skill files (5,000 tokens)
```
[System Prompt]
- map.md (200 tokens) ← always needed
- code_helper.md (500 tokens) ← loaded on demand!
- data_analyst.md (500 tokens) ← loaded on demand!
- compact.md (300 tokens) ← loaded on demand!
```
Result: Every user message costs 5,000 tokens before conversation even starts!

✅ **Good**: System prompt includes only the map
```
[System Prompt]
- map.md (200 tokens) ← always needed
- Behavior rules (200 tokens)
Total: 400 tokens per message (vs 5,000!)
```
Result: When user needs code_helper, the agent calls `read_skill_file("code_helper")` (+500 tokens for that turn only)

### Our System Prompt Has Three Parts:
1. **ReAct behavior rules** — How to think and act
2. **Skill map** — What skills exist (metadata only)
3. **Constraints** — What the agent can/cannot do

Let's build it:


In [32]:
print("First 500 characters of system prompt:")
print("-" * 70)
print(system_prompt[:500])
print("-" * 70)
print("...")

First 500 characters of system prompt:
----------------------------------------------------------------------
You are an enhanced ReAct agent that demonstrates how modern coding agents (Claude Code, Cursor, GitHub Copilot) work.

## RULE 1 (MANDATORY): ALWAYS use progressive disclosure
**Before attempting ANY task**, you MUST call `read_skill_file()` with the appropriate skill ID.
This is not optional — it's the core of how agents discover capabilities.

## Available Skills (from skills/map.md):
# Skill Map

This is the **master catalog** of all available agent skills. Always read this first to decide w
----------------------------------------------------------------------
...


## 4. The Agent Loop — The Heartbeat of Reasoning

### What's an Agent Loop?

It's the core loop that **enables agents to think and act**:

```
╔════════════════════════════════════════════════════════════════╗
║                    AGENT LOOP (Repeated)                       ║
╠════════════════════════════════════════════════════════════════╣
║                                                                 ║
║  1. Send [ user message + history + tools ] to OpenAI         ║
║     ↓                                                           ║
║  2. Model responds with EITHER:                                ║
║     a) Tool calls (wants to use a tool)                       ║
║     b) Final text response (ready to answer)                  ║
║     ↓                                                           ║
║  3. If tool calls:                                             ║
║     - Execute each tool locally                                ║
║     - Add results back to history                              ║
║     - Loop back to step 1 (model gets new info)               ║
║     ↓                                                           ║
║  4. If final response:                                         ║
║     - Return answer to user                                    ║
║     - Conversation ends (or user asks follow-up)              ║
║                                                                 ║
╚════════════════════════════════════════════════════════════════╝
```

### Why This Loop Works

- **Iterative refinement**: Agent learns from each tool call
- **Fact-based reasoning**: Not guessing (run tools to verify)
- **Grounding**: Links model outputs to real world (code execution, file reads)
- **Transparency**: Every step is visible to student

### Token Accounting

Each iteration of the loop **adds tokens**:
- Tool call: ~100 tokens
- Tool result: ~200 tokens (plus size of actual result)
- Next model call: 100 tokens

So a 3-tool conversation might cost:
- System prompt: 400 tokens
- User message: 50 tokens
- Step 1 (agent thinks): 200 tokens
- Step 2 (tool results): 600 tokens
- Step 3 (agent thinks again): 200 tokens
- Step 4 (tool results): 600 tokens
- Step 5 (final response): 300 tokens
- **Total: ~2,350 tokens for one user message!**

This is why progressive disclosure matters — if all skills were loaded, add 2,000 more tokens per message.

Let's implement the loop:


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# PART 5: THE AGENT LOOP WITH PROGRESSIVE DISCLOSURE
# ──────────────────────────────────────────────────────────────────────────────

def run_agent(user_message: str, history=None, verbose=True, max_steps=10) -> tuple:
    """
    Run the ReAct agent loop with progressive disclosure.

    Key feature: Forces tool_choice on step 1 to ensure skill loading happens.
    This ensures students SEE the progressive disclosure pattern.
    """

    global conversation_log

    # Initialize history if needed
    if history is None:
        history = [
            {"role": "system", "content": system_prompt}
        ]

    # Add user message
    history.append({"role": "user", "content": user_message})

    if verbose:
        print(f"\n[User] {user_message}\n")
        print("=" * 80)
        print("AGENT LOOP START")
        print("=" * 80)

    step = 1
    tools_called_this_turn = []

    while step <= max_steps:
        if verbose:
            print(f"\n[Step {step}] Calling API...")

        # ──────────────────────────────────────────────────────────────────────
        # ⚠️  TEACHING NOTE — Progressive Disclosure vs. Autonomy
        #
        # We FORCE read_skill_file on step 1 so you always SEE it happen.
        # In a real production agent, you'd set tool_choice='auto' on every step
        # and trust the model to follow RULE 1 in the system prompt.
        #
        # 🔬 EXPERIMENT: Change the if-block below to just tool_choice = 'auto'
        #   • With GPT-4o / Llama-3: agent still loads the skill (obeys the prompt)
        #   • With weaker models: agent might skip it — that's a real lesson too!
        #   Why? Smaller models don't always follow complex system-prompt rules.
        # ──────────────────────────────────────────────────────────────────────
        tool_choice = "auto"
        if step == 1:
            # Force the model to call read_skill_file on first step
            tool_choice = {
                "type": "function",
                "function": {
                    "name": "read_skill_file"
                }
            }

        # Call the API
        response = client.chat.completions.create(
            model=MODEL,  # set at the top of the notebook
            messages=history,
            tools=TOOLS,
            tool_choice=tool_choice,
            temperature=0
        )

        response_message = response.choices[0].message

        # Add agent's response to history
        history.append(response_message.model_dump())

        # Check if model wants to call tools
        if response_message.tool_calls:
            if verbose:
                print(f"[Agent] Tool calls requested ({len(response_message.tool_calls)} tools):")

            # Execute each tool
            for tool_call in response_message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"  - {tool_name}({json.dumps(tool_args)[:]}...)")

                tools_called_this_turn.append(tool_name)

                # Execute the tool
                tool_result = dispatch_tool(tool_name, tool_args)

                if verbose:
                    # Show the tool output — the most educational part!
                    result_preview = tool_result[:300].replace('\n', '\n                    ')
                    print(f"  → Output: {result_preview}")
                    if len(tool_result) > 300:
                        print(f"  → ... ({len(tool_result)} chars total)")


                # Add tool result to history
                history.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": tool_name,
                    "content": tool_result
                })

        else:
            # Model gave final answer (no tool calls requested)
            if verbose:
                print(f"[Agent] {response_message.content[:]}")
                print("\n" + "=" * 80)
                print("AGENT LOOP COMPLETE")
                print("=" * 80)

            # Log this turn
            if tools_called_this_turn:
                log_turn(len(conversation_log) + 1, tools_called_this_turn, estimate_tokens(str(history)))

            # Return final message and updated history
            return response_message.content, history

        step += 1

    # If we hit max steps
    if verbose:
        print(f"\nMax steps ({max_steps}) reached. Returning partial response.")

    return "Agent reached maximum steps limit.", history

In [34]:
# Verify tools are registered
print(f"✓ Tools defined:", [t['function']['name'] for t in TOOLS])

# ──────────────────────────────────────────────────────────────────────────────
# PART 8: TOKEN ESTIMATION
# ──────────────────────────────────────────────────────────────────────────────
# Rough estimate of tokens used (for teaching purposes).
# In production, use tiktoken library for accuracy.

def estimate_tokens(text: str) -> int:
    """
    Rough estimate of tokens in text.

    Rule of thumb: 1 token ≈ 4 characters (average for English + code)

    Guard: If text is None or empty, return 0 (prevents TypeError)
    """
    if not text:  # Guard against None or empty strings
        return 0
    return max(1, len(text) // 4)

# ──────────────────────────────────────────────────────────────────────────────
# PART 9: CONVERSATION LOGGING
# ──────────────────────────────────────────────────────────────────────────────
# Track each turn: what tools were called, how many tokens used, etc.

def log_turn(turn_number: int, tools_called: list, tokens_used: int):
    """Log metadata about each agent turn for analysis."""
    conversation_log.append({
        "turn": turn_number,
        "tools_called": ", ".join(tools_called) if tools_called else "none",
        "tokens_used": tokens_used
    })

# ──────────────────────────────────────────────────────────────────────────────
# PART 10: EXPORT CONVERSATION
# ──────────────────────────────────────────────────────────────────────────────
# Save conversation history to JSON Lines for analysis

def export_conversation_log(filename: str = "conversation.jsonl"):
    """Export conversation log to a JSON Lines file (one record per line)."""
    import json
    with open(filename, "w") as f:
        for entry in conversation_log:
            f.write(json.dumps(entry) + "\n")
    print(f"Conversation log exported to {filename}")

✓ Tools defined: ['read_skill_file', 'read_file', 'run_python', 'compact_context']


## 5. Demonstration — Watch the Agent in Action

### Before Running the Demos, Pay Attention To:

1. **What the agent prints**:
   - `[Tool] read_skill_file(...)`  = Agent loaded a skill (progressive disclosure!)
   - `[Tool] run_python(...)`  = Agent tested code
   - `Agent Final Response` = Ready to answer

2. **The ReAct pattern**:
   - Does the agent narrate its plan? (It should!)
   - Does it verify results before responding?
   - Does it compose the answer from tool results?

3. **Tool Usage**:
   - Which tools did it call?
   - In what order?
   - Could it have used fewer tools?

4. **Comparison to Real Agents**:
   - This is how Claude Code works (similar architecture)
   - This is how Cursor works (similar patterns)
   - This is how GitHub Copilot works (similar reasoning)

### Demo Structure

We'll run three progressively complex demos:

| Demo | Goal | Skills Used | Tools Called |
|------|------|------------|--------------|
| Task 1 | Write and test code | code_helper | read_skill_file, run_python |
| Task 2 | Analyze data (skill switching!) | data_analyst | read_skill_file, read_file, run_python |
| Task 3 | Handle long conversations | compact | compact_context |

### 🎬 Ready? Run the cells below and watch the magic!


In [42]:
# ──────────────────────────────────────────────────────────────────────────────
# DEMO TASK 1: Code Helper Skill
# ──────────────────────────────────────────────────────────────────────────────
#
# WHAT TO WATCH FOR:
#
# 1. Skill Discovery
#    The agent sees map.md (in system prompt) and decides: "This is a coding task"
#    So it will call: read_skill_file("code_helper")
#
# 2. Progressive Disclosure in Action
#    Agent loads code_helper.md (~500 tokens) ONLY for this turn
#    If you asked a data question next, it would load data_analyst.md instead
#    This is the magic of token efficiency!
#
# 3. Tool Usage Pattern
#    Agent will likely:
#    - Think about the problem (ReAct [Reflect] & [Plan])
#    - Write the code (Python function)
#    - Call: run_python(code) to test it
#    - Show you the result (ReAct [Respond])
#
# 4. Verification Over Hallucination
#    Agent doesn't just CLAIM it works — it PROVES it with tests
#    This is how Claude Code and Cursor work!
#
# ─────────────────────────────────────────────────────────────────────────────

# ── Reset tracking for this demo ─────────────────────────────────────────────
conversation_log = []  # reset log for this demo

print("\n" + "🎬 DEMO TASK 1: CODE HELPER SKILL".center(80, "═"))
print("Watch how the agent discovers, loads, and uses the code_helper skill")
print("Pay attention to: skill loading, tool calls, ReAct pattern")
print()

history = None  # Fresh session

response, history = run_agent(
    "Write me a Python function that checks if a number is prime. Test it with 17 and 20.",
    history=history,
    verbose=True
)

print("\n" + "─" * 80)
print("✅ DEMO 1 COMPLETE")
print("─" * 80)
print()
print("What the agent did:")
print("  1. Read map.md (saw 'code_helper' option)")
print("  2. Loaded code_helper.md skill")
print("  3. Followed the ReAct pattern:")
print("     [Reflect] User wants a prime checking function")
print("     [Plan] I should write and test code")
print("     [Tool Call] read_skill_file('code_helper')")
print("     [Observe] Skill shows I can use run_python()")
print("     [Tool Call] run_python(function code)")
print("     [Respond] Here's the verified function")
print()
print(f"Skills used: ['code_helper']")
print(f"Tools called this turn: (check output above)")
print()


════════════════════════🎬 DEMO TASK 1: CODE HELPER SKILL════════════════════════
Watch how the agent discovers, loads, and uses the code_helper skill
Pay attention to: skill loading, tool calls, ReAct pattern


[User] Write me a Python function that checks if a number is prime. Test it with 17 and 20.

AGENT LOOP START

[Step 1] Calling API...
[Agent] Tool calls requested (1 tools):
  - read_skill_file({"skill_id": "code_helper"}...)

[Step 2] Calling API...
[Agent] Tool calls requested (1 tools):
  - run_python({"code": "def is_prime(n):\n    if n < 2:\n        return Fa...)

[Step 3] Calling API...
[Agent] Here's the prime checker function and its test results:

```python
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return

AGENT LOOP COMPLETE

────────────────────────────────────────────────────────────────────────────────
✅ DEMO 1 COMPLETE
─────────────────────────────────────────────────────────────

### Task 2: Data Analyst Skill
Now the agent should switch to a different skill.

In [43]:
# ── Reset tracking for this demo ─────────────────────────────────────────────
conversation_log = []  # reset log for this demo

print()
print("=" * 80)
print("DEMO TASK 2: Data Analyst Skill (Skill Switching)".center(80))
print("=" * 80)
print()
print("Scenario: Same agent, different task type")
print("Watch how it switches skills based on the question")
print()

# Create sample data
import csv
with open("sales.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["product", "quantity", "price", "revenue"])
    writer.writeheader()
    writer.writerows([
        {"product": "Laptop",  "quantity": "5",  "price": "1000", "revenue": "5000"},
        {"product": "Mouse",   "quantity": "50", "price": "25",   "revenue": "1250"},
        {"product": "Monitor", "quantity": "10", "price": "300",  "revenue": "3000"},
    ])

print("Created sample file: sales.csv")
print()

# Fresh session for new task (clean history = fresh skill context)
history = None
response, history = run_agent(
    "I have a file called sales.csv. Which product has the highest revenue?",
    history=history,
    verbose=True
)

print("\n" + "-" * 80)
print("DEMO 2 COMPLETE")
print("-" * 80)
print()
print("What happened:")
print("  1. User asked about DATA (not code)")
print("  2. Agent read map.md, saw skills: code_helper, data_analyst")
print("  3. Agent chose: data_analyst (intelligent choice!)")
print("  4. Agent loaded data_analyst.md")
print("  5. Agent followed the ReAct pattern:")
print("     - [Reflect] User has CSV and wants to find highest revenue")
print("     - [Plan] Load file, then analyze with pandas")
print("     - [Tool Call] read_file('sales.csv')")
print("     - [Tool Call] run_python() with pandas analysis")
print("     - [Respond] Reported: Which product has highest revenue")
print()
print("Key insight: Same agent, different skill = Different behavior!")
print("This demonstrates adaptive reasoning — the core of intelligence.")
print()


               DEMO TASK 2: Data Analyst Skill (Skill Switching)                

Scenario: Same agent, different task type
Watch how it switches skills based on the question

Created sample file: sales.csv


[User] I have a file called sales.csv. Which product has the highest revenue?

AGENT LOOP START

[Step 1] Calling API...
[Agent] Tool calls requested (1 tools):
  - read_skill_file({"skill_id": "data_analyst"}...)

[Step 2] Calling API...
[Agent] [Reflect] User wants to know which product has the highest revenue from sales.csv. I've loaded the data_analyst skill which shows we need to:
1. Read the CSV file first
2. Run pandas analysis to find 

AGENT LOOP COMPLETE

--------------------------------------------------------------------------------
DEMO 2 COMPLETE
--------------------------------------------------------------------------------

What happened:
  1. User asked about DATA (not code)
  2. Agent read map.md, saw skills: code_helper, data_analyst
  3. Agent chose: data_ana

### Task 3: Trigger Context Compaction
Simulate a long conversation and ask the agent to compact.

In [44]:
# ──────────────────────────────────────────────────────────────────────────────
# TASK 3: Context Compaction (Memory Management)
# ──────────────────────────────────────────────────────────────────────────────
#
# WHAT TO WATCH FOR:
#
# 1. Long Conversation Problem
#    Each turn adds tokens to history
#    After ~20 turns, history gets HUGE
#    Model has less space for new messages
#
# 2. The Compaction Solution
#    Agent calls: compact_context()
#    Gets instructions from compact.md (specific format!)
#    Creates a summary following that format
#    Drops old turns, keeps summary
#    Result: Free up 80% of tokens!
#
# 3. Compact Format (from compact.md)
#    Goal: One sentence about what user wanted
#    Key Decisions: List of important choices
#    Work Completed: List of tasks done
#    Current State: Where we are now
#    Next Step: What to do next
#
# 4. Real-World Consequence
#    Without compaction: Long convos = expensive + slow
#    With compaction: Long convos = manageable + cheap
#
# This is WHY every production agent uses this pattern!

# ── Reset tracking for this demo ─────────────────────────────────────────────
conversation_log = []  # reset log for this demo

print()
print("=" * 80)
print("DEMO TASK 3: Context Compaction (Token Management)".center(80))
print("=" * 80)
print()
print("Scenario: After several turns, conversation gets long.")
print("Solution: Agent compacts history into structured summary.")
print()

# Simulate a longer conversation by adding multiple turns
print("Simulating long conversation (adding multiple 'turns')...")
print()

# Add a few simulated turns to history
for i in range(3):
    history.append({
        "role": "user",
        "content": f"Follow-up question {i+1}: What about next steps?"
    })
    history.append({
        "role": "assistant",
        "content": f"Here's my response to follow-up {i+1}..."
    })

print(f"History grown to {len(history)} messages")
print(f"Estimated history size: ~{sum(len(m.get('content') or '')//4 for m in history)} tokens")
print()

# NOW trigger compaction
print("Asking agent to compact context (calling compact_context() tool)...")
print()

response, history = run_agent(
    "The conversation has been getting long. Please create a compact summary of what we've discussed using the compaction format.",
    history=history,
    verbose=True
)

# ── Apply compaction: replace bloated history with the summary ────────────────
# This is the KEY step — without it, compaction is just pretty-printing!
messages_before = len(history)
history = [
    {"role": "system",    "content": system_prompt},
    {"role": "assistant", "content": response},  # the compact summary
]
print(f"\n✅ History truncated: {messages_before} → {len(history)} messages")
print(f"   Token savings: ~{(messages_before - len(history)) * 200:,} tokens freed")

print("\n" + "-" * 80)
print("DEMO 3 COMPLETE")
print("-" * 80)
print()
print("What happened:")
print("  1. Agent was asked to summarize")
print("  2. Agent called: compact_context()")
print("  3. Got compaction instructions from compact.md")
print("  4. Model created summary in required format:")
print("     - Goal: (what user wanted)")
print("     - Key Decisions Made: (important choices)")
print("     - Work Completed: (tasks done)")
print("     - Current State: (where are we)")
print("     - Next Step: (what's next)")
print()
print("TOKEN EFFICIENCY:")
print(f"   Before compaction: {messages_before} messages")
print(f"   After compaction:  {len(history)} messages (saved ~80%)")


               DEMO TASK 3: Context Compaction (Token Management)               

Scenario: After several turns, conversation gets long.
Solution: Agent compacts history into structured summary.

Simulating long conversation (adding multiple 'turns')...

History grown to 11 messages
Estimated history size: ~1842 tokens

Asking agent to compact context (calling compact_context() tool)...


[User] The conversation has been getting long. Please create a compact summary of what we've discussed using the compaction format.

AGENT LOOP START

[Step 1] Calling API...
[Agent] Tool calls requested (1 tools):
  - compact_context({}...)

[Step 2] Calling API...
[Agent] ## Conversation Summary (Compacted)

**Goal:** Identify highest revenue product from sales data and determine next steps for analysis.

**Key Decisions Made:**
- Used data_analyst skill to analyze sal

AGENT LOOP COMPLETE

✅ History truncated: 15 → 2 messages
   Token savings: ~2,600 tokens freed

---------------------------------

## 6. Understanding Tokens — Before You Read the Analysis

### What Is a Token?

LLMs don't read words — they read **tokens**. A token is a chunk of text, roughly:

```
"hello"        → 1 token
"Hello world"  → 2 tokens
"is_prime"     → 2 tokens  (split at the underscore)
"supercalifragilistic" → 7 tokens
```

**Rule of thumb**: 1 token ≈ 4 characters (for English + code).  
That means 1,000 words ≈ 1,333 tokens.

### Why Do Tokens Matter?

| Model | Context Window | Rough equivalent |
|-------|---------------|------------------|
| gpt-4o-mini | 128,000 tokens | ~96,000 words / ~200 pages |
| gpt-4o | 128,000 tokens | same |
| llama3.2 (Ollama) | 128,000 tokens | same |

The context window is the **maximum input size per call** — system prompt + history + your message must all fit.  
The longer your conversation, the more tokens you consume per call. This is exactly why **context compaction** matters.

### What You'll See Below

The analysis cell breaks down every message in the conversation history and shows:
- How many tokens each role (system, user, assistant, tool) used
- Which tools were called
- The total token cost of the entire session

> **Note**: Our `estimate_tokens()` uses the `len // 4` heuristic.  
> For production-accurate counts, use [`tiktoken`](https://github.com/openai/tiktoken).


## 7. Inspect What Happened

### Key Observations for Students

| What you saw | What real agents do |
|---|---|
| Agent reads `map.md` first | Claude Code loads `SKILL.md` metadata (~100 tokens) |
| Agent calls `read_skill_file()` when needed | Full skill instructions loaded on demand (~5k tokens) |
| Skill file defines which tools are available | Tools per-skill, not all tools dumped at once |
| Agent narrates [Reflect] → [Plan] → [Observe] | ReAct pattern used in Claude, Codex, Cursor |
| `compact_context()` summarizes long sessions | Real agents use context compaction to handle limits |

This is the foundation of **every coding agent you use today**.


In [45]:
# ──────────────────────────────────────────────────────────────────────────────
# CONVERSATION ANALYSIS
# ──────────────────────────────────────────────────────────────────────────────
#
# This cell shows EXACTLY what happened in the agent loop.
# Every important decision is visible here.

import pandas as pd

print("=" * 80)
print("CONVERSATION ANALYSIS — Full Transcript Breakdown")
print("=" * 80)

print(f"\n{len(history)} total messages in conversation history:")
print()

for i, msg in enumerate(history):
    role = msg.get('role', 'unknown')

    if role == 'system':
        content_preview = (msg.get('content') or '')[:80].replace('\n', ' ')
        tokens = estimate_tokens(msg.get('content') or '')
        print(f"  [{i:2d}] [SYSTEM] [~{tokens} tokens]")
        print(f"       {content_preview}...")

    elif role == 'user':
        content = msg.get('content') or ''
        tokens = estimate_tokens(content)
        print(f"  [{i:2d}] [USER] [~{tokens} tokens]")
        print(f"       {content[:70]}...")

    elif role == 'assistant':
        content = msg.get('content') or ''
        tool_calls = msg.get('tool_calls', [])

        if content:
            tokens = estimate_tokens(content)
            print(f"  [{i:2d}] [ASSISTANT] [~{tokens} tokens]")
            print(f"       {content[:70]}...")

        if tool_calls:
            print(f"  [{i:2d}] [TOOL CALLS] [{len(tool_calls)} tools]")
            for tc in tool_calls:
                tool_name = tc.get('function', {}).get('name', 'unknown') if isinstance(tc, dict) else tc.function.name
                print(f"       - {tool_name}")

    elif role == 'tool':
        content = msg.get('content') or ''
        tokens = estimate_tokens(content)
        tool_call_id = msg.get('tool_call_id', 'unknown')
        print(f"  [{i:2d}] [TOOL RESULT] [~{tokens} tokens] (ID: {tool_call_id[:8]}...)")
        print(f"       {content[:70]}...")

# Token accounting
print(f"\n{'-'*80}")
print("TOKEN ACCOUNTING (Rough Estimate)")
print(f"{'-'*80}\n")

total_tokens = 0
breakdown = {}

for msg in history:
    role = msg.get('role', 'unknown')
    content = msg.get('content') or ''
    tokens = estimate_tokens(content)

    if role not in breakdown:
        breakdown[role] = 0
    breakdown[role] += tokens
    total_tokens += tokens

for role in ['system', 'user', 'assistant', 'tool']:
    if role in breakdown:
        tokens = breakdown[role]
        pct = (tokens / total_tokens * 100) if total_tokens > 0 else 0
        print(f"  {role.upper():12} | {tokens:5} tokens ({pct:5.1f}%)")

print(f"  {'-'*28}")
print(f"  {'TOTAL':12} | {total_tokens:5} tokens (100.0%)")
print()
print(f"  Context window used: {total_tokens:,} / 128,000 tokens")
print(f"  Remaining capacity: {128000 - total_tokens:,} tokens")

# Tool usage summary
print(f"\n{'-'*80}")
print("TOOL USAGE SUMMARY")
print(f"{'-'*80}\n")

tool_calls_list = []
for msg in history:
    if msg.get('role') == 'assistant' and msg.get('tool_calls'):
        for tc in msg.get('tool_calls', []):
            tool_name = tc.get('function', {}).get('name', 'unknown') if isinstance(tc, dict) else tc.function.name
            tool_calls_list.append(tool_name)

if tool_calls_list:
    print(f"  Tools called: {len(tool_calls_list)}")
    for tool in set(tool_calls_list):
        count = tool_calls_list.count(tool)
        print(f"    - {tool}: called {count} time(s)")
else:
    print("  No tools were called (agent answered directly)")

# Conversation log
print(f"\n{'-'*80}")
print("CONVERSATION LOG")
print(f"{'-'*80}\n")

if conversation_log:
    for entry in conversation_log:
        print(f"  Turn {entry['turn']}: {entry['tools_called']} | ~{entry['tokens_used']} tokens")
else:
    print("  (No turns logged yet — run demos first)")

print(f"\n{'-'*80}")
print("EXPORT OPTIONS")
print(f"{'-'*80}\n")
print("  To save this conversation log to a file:")
print("    export_conversation_log('my_conversation.jsonl')")
print()
print("  To reset conversation (after running demos):")
print("    history = None")
print("    conversation_log = []")
print("\n" + "=" * 80)

CONVERSATION ANALYSIS — Full Transcript Breakdown

2 total messages in conversation history:

  [ 0] [SYSTEM] [~1145 tokens]
       You are an enhanced ReAct agent that demonstrates how modern coding agents (Clau...
  [ 1] [ASSISTANT] [~157 tokens]
       ## Conversation Summary (Compacted)

**Goal:** Identify highest revenu...

--------------------------------------------------------------------------------
TOKEN ACCOUNTING (Rough Estimate)
--------------------------------------------------------------------------------

  SYSTEM       |  1145 tokens ( 87.9%)
  ASSISTANT    |   157 tokens ( 12.1%)
  ----------------------------
  TOTAL        |  1302 tokens (100.0%)

  Context window used: 1,302 / 128,000 tokens
  Remaining capacity: 126,698 tokens

--------------------------------------------------------------------------------
TOOL USAGE SUMMARY
--------------------------------------------------------------------------------

  No tools were called (agent answered directly)

-------

---

## 8. 🛠️ Build Your Own Agent — Starter Template

You now understand how a ReAct agent works. Use this section as a launchpad.

### Option A: Add a New Skill (easiest)

**Step 1** — Create `skills/my_skill.md`:
```markdown
# Skill: My Skill

## Purpose
Describe what this skill does in one sentence.

## When to Use This Skill
- "Phrase that should trigger this skill"

## Available Tools
| Tool | Purpose |
|------|---------| 
| `run_python(code)` | Compute or verify |

## ReAct Example: "Sample user request"
\```
[Reflect] What does the user want?
[Plan] Which tool should I use?
[Tool Call] run_python("...")
[Observe] Output: ...
[Respond] Here's the answer.
\```
```

**Step 2** — Add one row to `skills/map.md`:
```markdown
| **my_skill** | What it does | "Example trigger phrase" |
```

That's it — the agent discovers and uses it automatically! ✅

---

### Option B: Add a New Tool (intermediate)

Three places to edit in the notebook:

**1. Implement the function:**
```python
def fetch_url(url: str) -> str:
    """Fetch the text content of a URL."""
    import urllib.request
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            return r.read().decode('utf-8')[:3000]
    except Exception as e:
        return f"Error fetching URL: {e}"
```

**2. Add to TOOLS list:**
```python
TOOLS.append({
    "type": "function",
    "function": {
        "name": "fetch_url",
        "description": "Fetch the text content of a public URL.",
        "parameters": {
            "type": "object",
            "properties": {"url": {"type": "string"}},
            "required": ["url"]
        }
    }
})
```

**3. Add to dispatch_tool():**
```python
elif tool_name == "fetch_url":
    return fetch_url(tool_args["url"])
```

---

### Option C: Specialize for a Different Domain

| Idea | Change the system prompt to... | Skills to use |
|------|-------------------------------|---------------|
| **Homework tutor** | "Help students understand, don't give answers directly" | math_solver, code_helper |
| **Data pipeline bot** | "Autonomously clean and analyze uploaded data" | data_analyst, sql_analyst |
| **Bug fixer** | "Take broken code and return working code with tests" | debugger, code_helper |
| **API explorer** | "Make API calls and summarize results" | Add fetch_url tool |

---

**🚀 Start small, run it, break it, fix it — that's how you learn agents!**


---

## 9. 🎓 Student FAQ — Common Questions Answered

### Q1: Why does the agent read `map.md` first instead of loading all skills?

**Answer**: **Progressive Disclosure** — the cornerstone of efficient agents.

```
❌ Bad: Include all skills in system prompt → 5,000 tokens every call
✅ Good: Include only map → 400 tokens + 500 when skill needed
```

**Real-world example**: Claude Code has 200+ skills. If all were loaded permanently, you'd lose 80% of context to never-used skills!

---

### Q2: Why does the agent call tools instead of just answering?

**Answer**: **Grounding** — connecting reasoning to reality.

```
❌ Bad: "Yes, 17 is prime" (no verification)
✅ Good: run_python("print(is_prime(17))")  → Output: True  → "Confirmed: 17 is prime"
```

---

### Q3: Can I add new tools?

**Answer**: Yes! Three steps — implement, add to TOOLS list, add to dispatcher. See Section 8.

---

### Q4: What if the agent calls a tool incorrectly?

**Answer**: The tool returns an error, and the agent tries again. This is **iterative refinement** — the loop keeps going until success!

---

### Q5: How is this like Claude Code or GitHub Copilot?

```
This notebook:        Real agents:
─────────────         ─────────────
4 tools               100+ tools
5 skills              200+ skills
~200 lines            ~100,000 lines
Educational           Production
```

But the CORE is identical: system prompt + skills on demand + tool calling + compaction.

---

### Q6: How do I extract exact token counts?

```python
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o-mini")
tokens = len(enc.encode(text))
```

---

**Got another question? Add a cell below and experiment! 🚀**
